In [ ]:
!pip install langchain langchain-groq langchain_community

In [ ]:
import datetime
import sqlite3
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langchain_community.chat_message_histories import SQLChatMessageHistory
from dotenv import load_dotenv
from typing import List, Optional, Tuple, Any
import gradio as gr

In [1]:
GROQ_API_KEY = ""

In [ ]:
DEFAULT_MODEL = "llama-3.3-70b-versatile"
DB_PATH = "/content/conversations.db"                        # Fix 3: explicit Colab path
DB_CONNECTION_STRING = f"sqlite:///{DB_PATH}"

In [ ]:
SYSTEM_MESSAGE = (
    "You are a diet planning assistant. Help users create balanced meal plans "
    "based on their goals, dietary needs, and preferences. Ask about one topic "
    "at a time to avoid overwhelming the user."
)

In [ ]:
class DietChatBot:
    def __init__(self, session_id: Optional[str] = None):
        self.model = ChatGroq(
            model=DEFAULT_MODEL,
            api_key=GROQ_API_KEY,
            temperature=0
        )
        self.system_msg = SystemMessage(content=SYSTEM_MESSAGE)
        self._initialize_session(session_id)
        self._initialize_history()

    def _initialize_session(self, session_id: Optional[str] = None) -> None:
        self.session_id = self._generate_session_id() if not session_id else session_id

    def _generate_session_id(self) -> str:
        timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        return f"Diet Chat - {timestamp}"

    def _initialize_history(self) -> None:
        self.history = SQLChatMessageHistory(
            session_id=self.session_id,
            connection=DB_CONNECTION_STRING
        )
        if not self.history.get_messages():
            self.history.add_message(self.system_msg)

    def get_response(self, user_message: str) -> str:
        if not (user_message := user_message.strip()):
            return ""
        self.history.add_message(HumanMessage(content=user_message))
        response = self.model.invoke(self.history.get_messages())
        self.history.add_message(AIMessage(content=response.content))
        return response.content

    def new_session(self) -> None:
        self._initialize_session()
        self._initialize_history()

    def load_session(self, session_id: str) -> None:
        if not session_id:
            return
        self._initialize_session(session_id)
        self._initialize_history()

    def get_messages(self) -> List[BaseMessage]:
        return self.history.get_messages()

    @staticmethod
    def get_previous_conversations() -> List[str]:
        query = """
        SELECT session_id FROM message_store
        GROUP BY session_id
        HAVING COUNT(*) > 1
        ORDER BY session_id DESC
        """
        with sqlite3.connect(DB_PATH) as conn:
            cursor = conn.cursor()
            cursor.execute(query)
            return [row[0] for row in cursor.fetchall()]

In [ ]:
class PersistentChatBotUI:
    def __init__(self, diet_chatbot):
        self.diet_chatbot = diet_chatbot

    def send_message_handler(self, user_message: str, history: list) -> Tuple[str, list]:
        user_message = user_message.strip()
        if user_message:
            ai_response = self.diet_chatbot.get_response(user_message)
            history.append(gr.ChatMessage(content=user_message, role="user"))
            history.append(gr.ChatMessage(content=ai_response, role="assistant"))
        return "", history

    def send_message_handler(self, user_message: str, history: list):
        """Stream response into Gradio chat UI."""
        user_message = user_message.strip()
        if not user_message:
            yield "", history
            return

        # Add user message immediately
        history.append(gr.ChatMessage(content=user_message, role="user"))
        history.append(gr.ChatMessage(content="", role="assistant"))
        yield "", history

        # Stream tokens into the last assistant message
        for token in self.diet_chatbot.get_response(user_message):
            history[-1].content += token
            yield "", history


    def new_session_handler(self) -> Tuple[list, Any]:
        self.diet_chatbot.new_session()
        return [], self.update_session_choices()

    def load_session_handler(self, session_id: str) -> Tuple[list, Any]:
        if not session_id:
            return [], self.update_session_choices()
        self.diet_chatbot.load_session(session_id)
        return self._format_messages_for_ui(
            self.diet_chatbot.get_messages()
        ), self.update_session_choices()

    def _format_messages_for_ui(self, messages: List[BaseMessage]) -> list:
        ui_messages = []
        for msg in messages[1:]:
            if isinstance(msg, HumanMessage):
                ui_messages.append(gr.ChatMessage(content=msg.content, role="user"))
            elif isinstance(msg, AIMessage):
                ui_messages.append(gr.ChatMessage(content=msg.content, role="assistant"))
        return ui_messages

    def update_session_choices(self) -> gr.update:
        choices = self.diet_chatbot.get_previous_conversations()
        if self.diet_chatbot.session_id not in choices:
            choices.insert(0, self.diet_chatbot.session_id)
        return gr.update(choices=choices, value=self.diet_chatbot.session_id)

    def create_ui(self) -> gr.Blocks:
        with gr.Blocks(theme=gr.themes.Soft(), title="Diet Planning Assistant with History") as interface:
            gr.Markdown("# Diet Planning Assistant")
            gr.Markdown("I can help you create balanced meal plans. Your conversations are saved!")

            chatbot = gr.Chatbot(height=600, type="messages")

            with gr.Row():
                with gr.Column(scale=3):
                    msg = gr.Textbox(
                        placeholder="Ask about nutrition or diet plans...",
                        show_label=False,
                    )
                with gr.Column(scale=1):
                    submit = gr.Button("💬 Send", variant="primary")

            with gr.Row():
                sessions = gr.Dropdown(
                    choices=self.diet_chatbot.get_previous_conversations(),
                    value=self.diet_chatbot.session_id,
                    label="Previous Conversations",
                    interactive=True,
                )
                new = gr.Button("🆕 New Chat", size="sm")

            submit.click(self.send_message_handler, [msg, chatbot], [msg, chatbot])
            msg.submit(self.send_message_handler, [msg, chatbot], [msg, chatbot])
            new.click(self.new_session_handler, None, [chatbot, sessions])
            sessions.change(self.load_session_handler, sessions, [chatbot, sessions])
            interface.load(self.update_session_choices, None, sessions)

        return interface

In [ ]:
# Launch
chatbot_ui = PersistentChatBotUI(DietChatBot())
interface = chatbot_ui.create_ui()
interface.launch(share=True)